# FYP — Reproducible 200-URL XAI Evaluation

**Scope:** held-out binary test URLs; 100 benign + 100 malicious; DistilBERT checkpoint; IG, LIME, SHAP; component occlusion, deletion effect, stability, timing; CSV and JSON exports.

**Before running:** edit `CHECKPOINT`, `TEST_CSV`, and `LABEL_COL` in Configuration. The notebook **does not train a model**. It will fail clearly if the files are missing. Run the 3-URL pilot first. For the complete experiment, set `PILOT=False`. On a GTX 1650, expect substantial runtime for 200 × 3 explanations, occlusion, and perturbations. Save/resume is enabled.

**Methodological cautions:** explanations target the **original predicted class**; component occlusion uses an empty component as a proxy baseline, not a causal ground truth; URL edits are synthetic and may not preserve real-world semantics or ground-truth labels. Stability results are separated by prediction flip. All attribution values are mapped to character spans before component aggregation. SHAP uses PartitionExplainer with a text masker; LIME uses URL token segmentation; IG uses embedding-layer integrated gradients. Their estimands differ, so compare rankings and normalized magnitudes rather than interpreting raw attribution magnitudes as interchangeable.

In [1]:
# Run once in the SAME environment as your fine-tuned checkpoint.
# %pip install -U "transformers>=4.44,<5" "torch>=2.2" "captum>=0.7" "lime>=0.2" "shap>=0.46" pandas numpy scipy tqdm


In [2]:

from pathlib import Path
import os, re, json, random, time, hashlib, warnings

import numpy as np
import pandas as pd
import torch
import shap

from tqdm.auto import tqdm
from scipy.stats import spearmanr
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import LayerIntegratedGradients
from lime.lime_text import LimeTextExplainer

# Reproducibility
SEED = 42

# Project paths
PROJECT_ROOT = Path(
    r"C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection"
)

CHECKPOINT = PROJECT_ROOT / "models" / "distilbert" / "final_model"
TEST_CSV = PROJECT_ROOT / "dataset" / "processed" / "test.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "xai_200"

# Dataset configuration
URL_COL = "clean_url"
LABEL_COL = "label_binary"

# Model configuration
MAX_LENGTH = 96

PILOT = False
N_PER_CLASS = 100
N_PERTURBATIONS = 2

LIME_SAMPLES = 150
SHAP_MAX_EVALS = 192
IG_STEPS = 16
DELETE_TOP_K = 2

# Set random seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Create results directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify paths
print("Device:", DEVICE)
print("Project:", PROJECT_ROOT)
print("Model:", CHECKPOINT)
print("Test CSV:", TEST_CSV)
print("Results:", OUTPUT_DIR)

assert CHECKPOINT.is_dir(), (
    f"Checkpoint not found: {CHECKPOINT}"
)

assert TEST_CSV.is_file(), (
    f"Held-out test CSV not found: {TEST_CSV}"
)

print("\nConfiguration loaded successfully!")


c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Project: C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection
Model: C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\models\distilbert\final_model
Test CSV: C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\dataset\processed\test.csv
Results: C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai_200

Configuration loaded successfully!


In [3]:
tokenizer = AutoTokenizer.from_pretrained(str(CHECKPOINT), use_fast=True)
assert tokenizer.is_fast, 'Fast tokenizer required for offset mapping'
model = AutoModelForSequenceClassification.from_pretrained(str(CHECKPOINT)).to(DEVICE).eval()
assert model.config.num_labels == 2, 'This notebook expects a binary classifier'
# IMPORTANT: verify this is the class mapping used DURING TRAINING.
print('id2label:',model.config.id2label)
assert MAX_LENGTH <= model.config.max_position_embeddings, 'MAX_LENGTH exceeds model capacity'

def batch_proba(texts, batch_size=8):
    texts=[str(t) for t in texts]
    if not texts: return np.empty((0,2))
    chunks=[]
    for k in range(0,len(texts),batch_size):
        batch=tokenizer(texts[k:k+batch_size],padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt')
        batch={key:val.to(DEVICE) for key,val in batch.items()}
        with torch.inference_mode(): out=model(**batch).logits.softmax(-1)
        chunks.append(out.cpu().numpy())
    return np.concatenate(chunks)


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7426.07it/s]

id2label: {0: 'Benign', 1: 'Malicious'}


## Fixed, balanced held-out sample

If your test file is not a previously saved, genuinely held-out partition, stop and regenerate a leakage-free split **without retraining on the evaluation URLs**. The notebook cannot independently prove the CSV is held out. Duplicates are removed by URL before sampling.

In [4]:
df=pd.read_csv(TEST_CSV)
assert URL_COL in df and LABEL_COL in df, f'Expected columns {URL_COL}, {LABEL_COL}; got {list(df.columns)}'
df=df[[URL_COL,LABEL_COL]].dropna().copy()
df[URL_COL]=df[URL_COL].astype(str).str.strip()
df=df[df[URL_COL].str.len()>0].drop_duplicates(subset=[URL_COL])
assert set(df[LABEL_COL].unique()).issubset({0,1}), 'Expected binary numeric labels 0/1'
assert (df[LABEL_COL].value_counts().reindex([0,1],fill_value=0)>=N_PER_CLASS).all(), 'Not enough unique test URLs in both classes'
sample=pd.concat([df[df[LABEL_COL]==k].sample(N_PER_CLASS,random_state=SEED) for k in (0,1)],ignore_index=True)
sample=sample.sample(frac=1,random_state=SEED).reset_index(drop=True)
sample['sample_id']=sample[URL_COL].map(lambda s:hashlib.sha256(s.encode()).hexdigest()[:16])
sample.to_csv(OUTPUT_DIR/'fixed_200_test_urls.csv',index=False)
work=sample.iloc[:3].copy() if PILOT else sample
print('Full sample:',sample[LABEL_COL].value_counts().to_dict(),'Running:',len(work))


Full sample: {0: 100, 1: 100} Running: 200


## URL component spans and deterministic perturbations

Use parsed URL boundaries. The hostname is split into subdomain and an approximate registrable domain using `tldextract` if installed; otherwise the full hostname is treated as domain (conservative fallback). We **never fetch** these URLs. Synthetic modifications affect the path/query/fragment only, to reduce risk of changing host identity; they are **not guaranteed label preserving**.

In [5]:
from urllib.parse import urlsplit
try:
 import tldextract
 _extract=tldextract.TLDExtract(suffix_list_urls=None)
except ImportError:
 _extract=None

def component_spans(url):
    # Character spans in the ORIGINAL URL; non-overlapping, ordered.
    s=str(url); out=[]
    try: p=urlsplit(s)
    except ValueError: return [('other',0,len(s))]
    scheme_end=s.find('://')
    host_start=(scheme_end+3) if scheme_end>=0 else (2 if s.startswith('//') else 0)
    authority_end=len(s)
    for ch in '/?#':
        ix=s.find(ch,host_start)
        if ix>=0:authority_end=min(authority_end,ix)
    authority=s[host_start:authority_end]
    host=p.hostname or ''
    host_ix=s.lower().find(host.lower(),host_start,authority_end) if host else -1
    if host_ix>=0:
        if _extract and '.' in host:
            ext=_extract(host)
            suffix_domain='.'.join(x for x in (ext.domain,ext.suffix) if x)
            cut=len(host)-len(suffix_domain) if suffix_domain else 0
            if cut>0:out.append(('subdomain',host_ix,host_ix+cut))
            out.append(('domain',host_ix+cut,host_ix+len(host)))
        else:out.append(('domain',host_ix,host_ix+len(host)))
    # scheme, port, credentials and delimiters are assigned 'other'
    path_start=authority_end
    q=s.find('?',path_start); f=s.find('#',path_start)
    path_end=min([x for x in (q,f,len(s)) if x>=0])
    if path_end>path_start:out.append(('path',path_start,path_end))
    if q>=0:out.append(('query',q, f if f>q else len(s)))
    if f>=0:out.append(('fragment',f,len(s)))
    out=sorted(out,key=lambda x:x[1]); merged=[]; prev=0
    for name,a,b in out:
        if a>prev:merged.append(('other',prev,a))
        if b>a:merged.append((name,a,b));prev=max(prev,b)
    if prev<len(s):merged.append(('other',prev,len(s)))
    return merged or [('other',0,len(s))]

def edit_url(url,kind):
    # Do not treat these edits as certified label-preserving.
    p=urlsplit(url)
    if kind==0:
        path=p.path + ('/' if not p.path.endswith('/') else '') + 'index'
        return p._replace(path=path).geturl()
    else:
        query=p.query+('&' if p.query else '')+'ref=1'
        return p._replace(query=query).geturl()

def remove_span(url,a,b):return url[:a]+url[b:]

def cos_sim(a,b):
    a=np.asarray(a,float);b=np.asarray(b,float)
    den=np.linalg.norm(a)*np.linalg.norm(b)
    return float(np.dot(a,b)/den) if den>1e-12 else np.nan

def spans_to_components(url,spans):
    # spans is list (start,end,attribution), signed; allocate by overlap.
    comps=component_spans(url); vals={name:0.0 for name,_,_ in comps}
    for a,b,v in spans:
        if b<=a:continue
        for name,x,y in comps:
            overlap=max(0,min(b,y)-max(a,x))
            if overlap:vals[name]+=float(v)*overlap/(b-a)
    return vals

def vectorize(comp):
    return np.array([comp.get(k,0.) for k in ('subdomain','domain','path','query','fragment','other')],dtype=float)


## Three XAI implementations

All methods explain the **probability of the original predicted class**. For IG we integrate from PAD-token embeddings, holding attention mask fixed. LIME uses whitespace-free URL tokens split by punctuation, then maps token coefficients to original string positions. SHAP uses its text masker and partitions; output tokens are aligned sequentially to original characters. If alignment fails, the notebook records an error instead of fabricating an attribution.

In [6]:
from collections import defaultdict
from functools import partial

def ig_explain(url,target):
    enc=tokenizer(url,return_tensors='pt',truncation=True,max_length=MAX_LENGTH,return_offsets_mapping=True)
    offsets=enc.pop('offset_mapping')[0].tolist()
    ids=enc['input_ids'].to(DEVICE);mask=enc['attention_mask'].to(DEVICE)
    baseline=torch.full_like(ids,tokenizer.pad_token_id)
    baseline[:,0]=ids[:,0];baseline[:,-1]=ids[:,-1]
    def forward(input_ids,attention_mask):return model(input_ids=input_ids,attention_mask=attention_mask).logits.softmax(-1)
    lig=LayerIntegratedGradients(forward,model.get_input_embeddings())
    attr=lig.attribute(ids,baselines=baseline,additional_forward_args=(mask,),target=int(target),n_steps=IG_STEPS,internal_batch_size=4)
    vals=attr.sum(dim=-1).detach().cpu().numpy()[0]
    return [(a,b,float(v)) for (a,b),v in zip(offsets,vals) if b>a]

# Match alphanumeric runs OR individual punctuation marks; URL delimiters remain visible.
URL_TOKEN_RE=re.compile(r'[A-Za-z0-9]+|[^A-Za-z0-9\s]')
lime_explainer=LimeTextExplainer(class_names=['benign','malicious'],split_expression=lambda text: URL_TOKEN_RE.findall(text),bow=False,random_state=SEED)

def lime_explain(url,target):
    # LIME's regex split_expression normally denotes delimiters, not tokens.
    # Use an explicit tokenizer function that returns token strings.
    exp=lime_explainer.explain_instance(url,batch_proba,num_features=300,num_samples=LIME_SAMPLES,labels=(int(target),))
    words = exp.domain_mapper.indexed_string.as_list
    # Token indices refer to their order in the original string, not feature ranking.
    positions=[(m.start(),m.end(),m.group()) for m in URL_TOKEN_RE.finditer(url)]
    if len(words)!=len(positions) or any(str(w)!=tok for w,(_,_,tok) in zip(words,positions)):
        raise ValueError('LIME tokenizer and character positions do not align')
    return [(positions[idx][0],positions[idx][1],float(weight)) for idx,weight in exp.as_map()[int(target)]]

# SHAP expects callable batch -> Nx2 probabilities.
shap_masker=shap.maskers.Text(tokenizer,mask_token=tokenizer.mask_token)
shap_explainer=shap.Explainer(batch_proba,shap_masker,algorithm='partition',output_names=['benign','malicious'])

def shap_explain(url,target):
    result=shap_explainer([url],max_evals=SHAP_MAX_EVALS)
    vals=np.asarray(result.values[0]);tokens=list(result.data[0])
    if vals.ndim==2: vals=vals[:,int(target)]
    if len(tokens)!=len(vals):raise ValueError('SHAP token/value length mismatch')
    # SHAP tokenizer pieces may include WordPiece prefixes and whitespace.
    spans=[];cursor=0
    for tok,v in zip(tokens,vals):
        t=str(tok).replace('##','').replace('Ġ',' ').replace('▁',' ')
        if not t.strip() or t in (tokenizer.cls_token,tokenizer.sep_token,tokenizer.pad_token,tokenizer.mask_token):continue
        pos=url.find(t,cursor)
        if pos<0:pos=url.find(t.strip(),cursor);t=t.strip()
        if pos<0:raise ValueError(f'SHAP token alignment failed: {tok!r}')
        spans.append((pos,pos+len(t),float(v)));cursor=pos+len(t)
    return spans

EXPLAINERS={'IG':ig_explain,'LIME':lime_explain,'SHAP':shap_explain}


## Faithfulness, stability, runtime and resume

- **Occlusion agreement:** Spearman rank correlation between absolute component attributions and absolute changes in original-class probability when each component is removed. Undefined if constant rankings.
- **Deletion effect:** probability drop after removing the two highest-attribution components (signed positive contributions ranked first). This is a simple deletion proxy, **not** a full deletion AUC.
- **Stability:** cosine similarity between six-component attribution vectors and top-component agreement, using the same original target class. Also record whether the predicted class flips.
- **Runtime:** wall-clock time for explanation only, with CUDA synchronization. Occlusion/perturbation inference is measured separately as evaluation overhead.
- **Checkpointing:** one JSONL record per URL × method, append immediately; reruns skip completed combinations. Errors are logged separately and can be retried.

In [7]:
def timed(fn,*args):
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    t=time.perf_counter();result=fn(*args)
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    return result,time.perf_counter()-t

def evaluate_one(url,label,method):
    probs=batch_proba([url])[0];pred=int(np.argmax(probs));target=pred
    spans,explain_seconds=timed(EXPLAINERS[method],url,target)
    comp=spans_to_components(url,spans);names=list(comp.keys())
    # Component occlusion reference (original target class).
    parts=component_spans(url)
    edited=[remove_span(url,a,b) for _,a,b in parts]
    t0=time.perf_counter();p_occ=batch_proba(edited)[:,target]
    occ={name:float(abs(probs[target]-p)) for (name,_,_),p in zip(parts,p_occ)}
    # Components can repeat in unusual URLs; group conservatively by max.
    # Only evaluate a name once for ordinary URLs; duplicates are flagged.
    duplicate_names=len(set(n for n,_,_ in parts))!=len(parts)
    if duplicate_names:
        occ={}
        for name in names:
            pieces=[(a,b) for n,a,b in parts if n==name]
            candidate=url
            for a,b in sorted(pieces,reverse=True):candidate=remove_span(candidate,a,b)
            occ[name]=float(abs(probs[target]-batch_proba([candidate])[0,target]))
    x=np.array([abs(comp.get(k,0)) for k in names]);y=np.array([occ.get(k,0) for k in names])
    valid=(len(names)>=2 and np.std(x)>1e-12 and np.std(y)>1e-12)
    rank_agreement=float(spearmanr(x,y).statistic) if valid else np.nan
    positives=sorted(parts,key=lambda p:comp.get(p[0],0),reverse=True)
    selected=positives[:DELETE_TOP_K]
    deleted=url
    for _,a,b in sorted(selected,key=lambda p:p[1],reverse=True):deleted=remove_span(deleted,a,b)
    deletion_drop=float(probs[target]-batch_proba([deleted])[0,target])
    stable_cos=[];stable_top=[];flips=[];perturb_times=[]
    for k in range(N_PERTURBATIONS):
        modified=edit_url(url,k)
        p2=batch_proba([modified])[0];flip=int(np.argmax(p2)!=pred)
        mod_spans,elapsed=timed(EXPLAINERS[method],modified,target)
        mod_comp=spans_to_components(modified,mod_spans)
        stable_cos.append(cos_sim(vectorize(comp),vectorize(mod_comp)))
        top1=max(comp,key=lambda z:abs(comp[z]))
        top2=max(mod_comp,key=lambda z:abs(mod_comp[z]))
        stable_top.append(int(top1==top2));flips.append(flip);perturb_times.append(elapsed)
    return dict(sample_id=hashlib.sha256(url.encode()).hexdigest()[:16],url=url,true_label=int(label),pred_label=pred,
                original_class_probability=float(probs[target]),method=method,explain_seconds=explain_seconds,
                evaluation_overhead_seconds=time.perf_counter()-t0,
                occlusion_spearman=rank_agreement,deletion_probability_drop=deletion_drop,
                mean_cosine_similarity=float(np.nanmean(stable_cos)) if np.isfinite(stable_cos).any() else np.nan,
                top_component_stability=float(np.mean(stable_top)),prediction_flip_rate=float(np.mean(flips)),
                perturb_explanation_seconds=float(np.sum(perturb_times)),
                component_attributions=json.dumps(comp),occlusion_effects=json.dumps(occ),
                n_components=len(names),n_perturbations=N_PERTURBATIONS)

records_path=OUTPUT_DIR/('pilot_results.jsonl' if PILOT else 'url_method_results.jsonl')
error_path=OUTPUT_DIR/('pilot_errors.jsonl' if PILOT else 'errors.jsonl')
completed=set()
if records_path.exists():
    for line in records_path.read_text(encoding='utf-8').splitlines():
        try:
            r=json.loads(line);completed.add((r['sample_id'],r['method']))
        except Exception:pass
for _,r in tqdm(work.iterrows(),total=len(work),desc='URLs'):
    url=str(r[URL_COL]);sid=r['sample_id']
    for method in EXPLAINERS:
        if (sid,method) in completed:continue
        try:
            output=evaluate_one(url,r[LABEL_COL],method)
            with records_path.open('a',encoding='utf-8') as f:f.write(json.dumps(output,ensure_ascii=False)+'\n')
            completed.add((sid,method))
        except Exception as exc:
            with error_path.open('a',encoding='utf-8') as f:f.write(json.dumps(dict(sample_id=sid,method=method,error=repr(exc)))+'\n')
            print('ERROR:',method,sid,repr(exc))
print('Results saved to:',records_path)


URLs: 100%|██████████| 200/200 [43:40<00:00, 13.10s/it]

Results saved to: C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai_200\url_method_results.jsonl


## Export report-ready CSV files and configuration

In [8]:
if records_path.exists() and records_path.stat().st_size:
    results=pd.read_json(records_path,lines=True)
    results=results.drop_duplicates(subset=['sample_id','method'],keep='last')
    results.to_csv(OUTPUT_DIR/('pilot_per_url.csv' if PILOT else 'per_url_method_results.csv'),index=False)
    metrics=['occlusion_spearman','deletion_probability_drop','mean_cosine_similarity','top_component_stability','prediction_flip_rate','explain_seconds','evaluation_overhead_seconds']
    summary=results.groupby('method')[metrics].agg(['count','mean','std','median'])
    summary.columns=['_'.join(c) for c in summary.columns]
    summary.to_csv(OUTPUT_DIR/('pilot_summary.csv' if PILOT else 'summary_by_method.csv'))
    display(summary.round(4))
    print('Successful unique URLs by method:',results.groupby('method').sample_id.nunique().to_dict())
    if not PILOT:
        assert all(results.groupby('method').sample_id.nunique().reindex(EXPLAINERS.keys(),fill_value=0)==200), 'Incomplete: do NOT report as a 200-URL evaluation yet'
else:print('No successful evaluations. Inspect errors JSONL and alignment/configuration.')
config={'seed':SEED,'checkpoint':str(CHECKPOINT.resolve()),'test_csv':str(TEST_CSV.resolve()),'max_length':MAX_LENGTH,'pilot':PILOT,'n_per_class':N_PER_CLASS,'n_perturbations':N_PERTURBATIONS,'lime_samples':LIME_SAMPLES,'shap_max_evals':SHAP_MAX_EVALS,'ig_steps':IG_STEPS,'device':str(DEVICE),'torch':torch.__version__,'shap':shap.__version__}
(OUTPUT_DIR/('pilot_config.json' if PILOT else 'config.json')).write_text(json.dumps(config,indent=2),encoding='utf-8')


,occlusion_spearman_count,occlusion_spearman_mean,occlusion_spearman_std,occlusion_spearman_median,deletion_probability_drop_count,deletion_probability_drop_mean,deletion_probability_drop_std,deletion_probability_drop_median,mean_cosine_similarity_count,mean_cosine_similarity_mean,...,prediction_flip_rate_std,prediction_flip_rate_median,explain_seconds_count,explain_seconds_mean,explain_seconds_std,explain_seconds_median,evaluation_overhead_seconds_count,evaluation_overhead_seconds_mean,evaluation_overhead_seconds_std,evaluation_overhead_seconds_median
method,,,,,,,,,,,,,,,,,,,,,
IG,193,0.3964,0.7740,1.0,200,0.6211,0.4685,0.9772,200,0.8517,...,0.1567,0.0,200,0.3748,0.1695,0.3279,200,0.8793,0.3647,0.7887
LIME,193,0.8295,0.3561,1.0,200,0.7308,0.4202,0.9773,200,0.9117,...,0.1567,0.0,200,2.3156,1.0532,2.1295,200,5.2522,2.0402,4.9169
SHAP,193,0.6891,0.5082,1.0,200,0.7869,0.3818,0.9773,200,0.9690,...,0.1567,0.0,200,1.3936,0.7908,1.1980,200,3.0277,1.3567,2.5838


Successful unique URLs by method: {'IG': 200, 'LIME': 200, 'SHAP': 200}


487

## Validation before reporting

1. Confirm the checkpoint maps class index 0 to benign and 1 to malicious. If not, fix mapping/labels **before** evaluating.
2. Confirm the test CSV was excluded from training and tuning; check URL overlap after normalization.
3. Inspect the first 3 URLs' component spans and token alignments. **Do not suppress alignment errors.**
4. Compare IG completeness delta and LIME/SHAP sensitivity to explanation budgets on a small subset.
5. Record all errors and success counts; if a method fails for some URLs, do not silently compare methods on different sets. Restrict comparisons to paired successful URLs and report exclusions.
6. Do not claim `1.0` scores prove perfect faithfulness or causal correctness. Report descriptive statistics and uncertainty. Consider paired bootstrap CIs if time permits.
7. Distinguish synthetic-edit stability from real-world adversarial robustness. A modified URL may have different meaning or true label.
8. For the MVP, use the same checkpoint and tokenizer, validate inputs, **never automatically visit user-supplied URLs**, and label confidence as model probability rather than guaranteed safety.